#### 1584. Min Cost to Connect All Points

* https://leetcode.com/problems/min-cost-to-connect-all-points/description/


In [ ]:
# Eager prims - TC - O(n^2)
# SC - O(n)
### Use Kruskal's algo when graph is sparse and Prims when graph is dense
### Kruskal time complexity - E log E
### if graph is dense, E ~ V^2 so it becomes E log V^2, each node connected to almost every other node
### Prims time complexity - E log V

### As a rule of thumb, if edge weights are given use Kruskal's, unless specified that the graph is dense use Kruskals else Prims


from typing import List

class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        """
        Computes the minimum cost to connect all points using Prim's Algorithm.

        Time Complexity: O(n^2)
        Space Complexity: O(n)
        """

        n: int = len(points)

        # Tracks whether a node is already included in MST
        in_mst: List[bool] = [False] * n

        # Minimum cost to connect each node to the MST
        min_cost: List[int] = [float('inf')] * n
        min_cost[0] = 0  # Start from node 0

        total_cost: int = 0

        for _ in range(n):
            # Step 1: Pick the node with the smallest connection cost
            current_node: int = -1
            current_min_cost: int = float('inf')

            for node in range(n):
                if not in_mst[node] and min_cost[node] < current_min_cost:
                    current_min_cost = min_cost[node]
                    current_node = node

            # Step 2: Add it to MST
            in_mst[current_node] = True
            total_cost += current_min_cost

            # Step 3: Update the cost of connecting remaining nodes
            x1, y1 = points[current_node]

            for neighbor in range(n):
                if not in_mst[neighbor]:
                    x2, y2 = points[neighbor]
                    manhattan_distance = abs(x1 - x2) + abs(y1 - y2)

                    # Relaxation step
                    if manhattan_distance < min_cost[neighbor]:
                        min_cost[neighbor] = manhattan_distance

        return total_cost

In [ ]:
# Prim's Algorithm - Lazy Prims as we are pushing all nodes and edges to minheap hence increasing the time complexity to n^2 log n
# TC - O(n^2 log n)
# Use Min heap to store cost and destination so min cost node is returned for each dest


import heapq
class Solution:
    def minCostConnectPoints(self, points) -> int:
        minheap = [(0, 0)] # starting from 0th node and cost to itself is 0
        n = len(points)
        visited = set()
        res = 0

        while len(visited) < n:
            cost, dest = heapq.heappop(minheap)
            if dest in visited:
                continue

            visited.add(dest)
            res += cost

            for i in range(n):
                if i not in visited:
                    duration = abs(points[dest][0] - points[i][0]) + abs(points[dest][1] - points[i][1]) # Manhattan Dist
                    heapq.heappush(minheap, (duration, i))

        return res


Solution().minCostConnectPoints([[3,12],[-2,5],[-4,1]])

18

Up to O(n²) edges considered

Each heap operation: O(log n)

Dominant term: O(n² log n)

In [ ]:
# Kruskals Solution
# Less Optimal because of a Dense Graph therefore time complexity - O(e log e) i.e e log v^2

from typing import List


class UnionFind:
    """
    Disjoint Set (Union-Find) with path compression and union by rank.
    """

    def __init__(self, size: int) -> None:
        self.parent = list(range(size))
        self.rank = [0] * size
        self.components = size  # Useful for early stopping

    def find(self, node: int) -> int:
        """Find root with path compression."""
        if self.parent[node] != node:
            self.parent[node] = self.find(self.parent[node])
        return self.parent[node]

    def union(self, u: int, v: int) -> bool:
        """
        Union by rank.
        Returns True if union happened (no cycle),
        False if already connected.
        """
        root_u = self.find(u)
        root_v = self.find(v)

        if root_u == root_v:
            return False  # cycle

        # Union by rank
        if self.rank[root_u] < self.rank[root_v]:
            self.parent[root_u] = root_v
        elif self.rank[root_u] > self.rank[root_v]:
            self.parent[root_v] = root_u
        else:
            self.parent[root_v] = root_u
            self.rank[root_u] += 1

        self.components -= 1
        return True


class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        """
        Kruskal's Algorithm to compute MST.

        Time Complexity:
            - Building edges: O(n^2)
            - Sorting edges: O(n^2 log n)
            - Union-Find ops: ~O(n^2 α(n))

            Overall: O(n^2 log n)

        Space Complexity:
            O(n^2) for edge list
        """

        n: int = len(points)
        edges = []

        # Step 1: Build all edges (complete graph)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                cost = abs(x1 - x2) + abs(y1 - y2)
                edges.append((cost, i, j))

        # Step 2: Sort edges by cost
        edges.sort(key=lambda edge: edge[0])

        uf = UnionFind(n)
        total_cost = 0
        edges_used = 0

        # Step 3: Process edges in increasing order
        for cost, u, v in edges:
            if uf.union(u, v):
                total_cost += cost
                edges_used += 1

                # Early stopping: MST complete
                if edges_used == n - 1:
                    break

        return total_cost